# Timeseries

## Introduction

This Notebook gives an overview of how temporal data (i.e. timeseries) is handled in Odeon.

In Odeon, temporal data is structured as follows:
- each branch is valid for exactly one year
- all data is either valid for the whole year (= scalar data) or is a timeseries with one value per hour for the branch's year (=temporal data)
- if the year is a leap year, the timeseries has 366⋅24 = 8784 values, otherwise it has 365⋅24 = 8760 values

To store temporal data in Odeon, the class `Temporal` is used. Per class, it is fixed which data is stored in scalar datatypes (e.g. `float`, `int`, `str`) and which data is stored in `Temporal` datatypes.

`Temporal`s allow the following features:
- arithmetic operations with scalars and other `Temporal`s
- split storage of relative timeseries (summing up to 1.0) and total (i.e. annual) value
- possibility to store constant values to indicate that the value is constant over time and to save memory and processing time
- referencing the timeseries of another `Temporal` object to avoid data duplication, allowing for a factor to scale the referenced timeseries
- swapping of timeseries data of a `Temporal` from main memory to disk and back to save memory

## Timeindex of a branch

When creating a branch, a timeindex is automatically created for the branch. The timeindex is a `pandas.DatetimeIndex` with one entry per hour of the year of the branch. The timeindex can be accessed via `branch.timeindex`:

In [1]:
# create a branch for year 2020 (leap year)
from odeon.model import Branch, Building, HeatDemand

branch = Branch(name="branch2020", year=2020)

# the branch has a timeindex with hourly resolution:
print(f"length: {len(branch.timeindex)}, start: {branch.timeindex[0]}, end: {branch.timeindex[-1]}")

configured logger with file /ceph/home/rob84552/repos/odeon/odeon/config/logging.yml
length: 8784, start: 2020-01-01 00:00:00, end: 2020-12-31 23:00:00


## Basics: Creating and using `Temporal`s

### Basic attributes: Relative and absolute series, total, fix, timeindex

In [2]:
from odeon.model import Temporal
import pandas as pd
import numpy as np

# create a series of a full year in hourly resolution, random values:
timeseries = pd.Series(data=np.random.rand(8760), index=pd.date_range("2021-01-01", periods=8760, freq="h"))

# same thing with rangeindex:
series = timeseries.reset_index(drop=True)

# create a Temporal from the series:
print("series:")
t_series = Temporal(series=series)
print(t_series) 
# the printout indicates that a total and a series are present (tot=... srs)

# ...or from a timeseries:
print("\ntimeseries:")
t_timeseries = Temporal(timeseries=timeseries)
print(t_timeseries)
# the printout indicates that a total, a series and a timeindex are present (tot=... idx srs)

# ...or from a total value:
print("\ntotal:")
t_total = Temporal(total=1000)
print(t_total)
# the printout indicates that only a total is present (tot=...)

# ...or from a constant value ("fix"):
print("\nfix:")
t_fix = Temporal(fix=5)
print(t_fix)
# the printout indicates that only a fix is present (fix=...). 
# We can't calculate a total or series from a fix without knowing the timeindex

# ...or from a relative series:
print("\nrelative series:")
shape = series
t_relative = Temporal(shape=shape)
print(t_relative)
# the printout indicates that only a relative series is present (rel)

# the relative series has been scaled to sum to 1.0:
print(f"sum of relative series: {t_relative.shape.sum()}")

series:
Temporal(id=None (tot=4388.192321804767) srs (shp))

timeseries:
Temporal(id=None (tot=4388.192321804767) idx srs (shp))

total:
Temporal(id=None tot=1000)

fix:
Temporal(id=None fix=5)

relative series:
Temporal(id=None shp)
sum of relative series: 0.9999999999999999


### `Temporal`s as attributes of objects

In [3]:
from odeon.model import Branch, Building, HeatDemand

# create an object that can hold a Temporal as attribute:
heat_demand = HeatDemand()

print("heat_demand.input_flow before setting:")
print(heat_demand.input_flow)
# at this point, the input_flow is an empty Temporal (no attributes set)
# but has a parent, indicated by (par empty) in the printout

# the input flow expects a temporal. We have different options to set it:
print("\nheat_demand.input_flow after setting:")
heat_demand.input_flow = 5 # will set a fix of 5
heat_demand.input_flow = Temporal(fix=5) # same as above
heat_demand.input_flow = Temporal(total=1000) # will set a total of 1000
heat_demand.input_flow = series # will set a series
heat_demand.input_flow = Temporal(series=series) # will set a series
print(heat_demand.input_flow)


heat_demand.input_flow before setting:
Temporal(id=8 par empty)

heat_demand.input_flow after setting:
Temporal(id=13 (tot=4388.192321804767) par srs (shp))


Strictly speaking, `input_flow` is not an attribute of `HeatDemand` but rather a property accessing deeper structures, as we can see from the object's attribute "_timeseries_attributes":

In [4]:
print(heat_demand._temporal_attributes) # no temporal attributes in this class
print(heat_demand.input_socket.link._temporal_attributes) # input_socket has a temporal attribute "input_flow"

[]
['_flow']


### Synchronization of timeindex with branch year

When a `Temporal` is set as an attribute to an objects which is part of a branch, the temporal will receive the branch's timeindex as implicit timeindex. An explicit timeindex is not allowed in this case.

In [5]:
# create an object in the branch with a Temporal attribute:
print("without branch:")
heat_demand = HeatDemand()
heat_demand.input_flow = 5 # will set fix only
print(heat_demand.input_flow)

print("\nwith branch:")
branch.add_objects(heat_demand)
print(heat_demand.input_flow)
# the printout indicates that a fix is present (fix=...) and that the temporal 
# has received an implicit timeindex from the branch (idx)

without branch:
Temporal(id=22 fix=5.0 par)

with branch:
Temporal(id=24 (tot=43920.0) fix=5.0 par (idx) (srs) (shp))


The timeindex can now be accessed from the temporal.
Series, timeseries and relative series can be calculated and are available as expected:

In [6]:
print("timeindex:")
print(f"length: {len(heat_demand.timeindex)}, start: {heat_demand.timeindex[0]}, end: {heat_demand.timeindex[-1]}")

print("\nseries:")
print(heat_demand.input_flow.series[:3])

print("\ntimeseries:")
print(heat_demand.input_flow.timeseries[:3])

print("\ntotal:")
print(heat_demand.input_flow.total)

print("\nshape:")
print(heat_demand.input_flow.shape[:3])

print("\ntimeshape:")
print(heat_demand.input_flow.timeshape[:3])

timeindex:
length: 8784, start: 2020-01-01 00:00:00, end: 2020-12-31 23:00:00

series:
0    5.0
1    5.0
2    5.0
dtype: float64

timeseries:
2020-01-01 00:00:00    5.0
2020-01-01 01:00:00    5.0
2020-01-01 02:00:00    5.0
Freq: h, dtype: float64

total:
43920.0

shape:
0    0.000114
1    0.000114
2    0.000114
dtype: float64

timeshape:
2020-01-01 00:00:00    0.000114
2020-01-01 01:00:00    0.000114
2020-01-01 02:00:00    0.000114
Freq: h, dtype: float64


### Manipulating `Temporal`s

## Arithmetic operations with `Temporal`s

## Referencing between `Temporal`s

## Swapping of `Temporal`s

As temporal data can be quite large, it is possible to swap the data of a `Temporal` from main memory to disk and back. This requires a branch and a project to be present, and the project needs to have a `FileAdapter`.

In [7]:
from odeon.model import Project
from odeon.io import FileAdapter

# create a temporary directory:
import tempfile
temp_dir = tempfile.TemporaryDirectory()

project = Project(
    file_adapter=FileAdapter(dir=temp_dir.name),
)
branch = Branch(name="branch2020", year=2020, project=project)

INFO enable.odeon.io.file_adapter: Set file manager directory to /tmp/tmpd6tv4p00


The file adapter needs to have a directory set, which is done via `project.file_adapter.dir = "some/directory"`.
To this directory, temporals of the project will write their data when swapping to disk in HDF5 files.

In [8]:
# add an object with a Temporal attribute to the branch:
heat_demand = HeatDemand()
branch.add_objects(heat_demand)

# set a series to the temporal attribute:
series = pd.Series(data=np.random.rand(8784), index=pd.RangeIndex(0, 8784)) # leap year!
heat_demand.input_flow = series
print(heat_demand.input_flow)

Temporal(id=36 (tot=4347.859696695891) par (idx) srs (shp))


### Manual mode

We can tell each timeseries individually to be swapped to disk or loaded back to memory via the attribute `swap_mode`, which can be set to `"loaded"` or `"swapped"`:

In [9]:
# swap to disk:
heat_demand.input_flow.swap_mode = "swapped"

Temporal data will be stored in one HDF file per branch. The file name will contain the branch ID to ensure uniqueness:

In [10]:
# print the content of the temporary directory of the file adapter:
import os
print(os.listdir(project.file_adapter.dir))
# file size:
print(f"file size: {os.path.getsize(os.path.join(project.file_adapter.dir, f'branch_{branch.id}.hdf')) / 1024:.2f} kB")

['branch_25.hdf']
file size: 68.29 kB


In [11]:
# load back to memory:
heat_demand.input_flow.swap_mode = "loaded"

# print file size again:
print(f"file size: {os.path.getsize(os.path.join(project.file_adapter.dir, f'branch_{branch.id}.hdf')) / 1024:.2f} kB")

file size: 68.29 kB


As we see, the file size hasn't changed. When loading back to memory, data won't get removed from disk automatically.

We can tell the file manager to clean the files:

In [12]:
project.file_adapter.clean()

# print file size again:
print(f"file size: {os.path.getsize(os.path.join(project.file_adapter.dir, f'branch_{branch.id}.hdf')) / 1024:.2f} kB")

file size: 0.78 kB


### Using `TemporalManager`

By assigning an instance of `TemporalManager` to our project, we can tell all `Temporal`s of the project to be swapped to disk or loaded back to memory at once:

In [13]:
from odeon.io import TemporalManager

project.temporal_manager = TemporalManager()
project.temporal_manager.set_swap_mode("swapped")

# swap mode of temporal will have changed:
print(heat_demand.input_flow.swap_mode)

swapped


A `TemporalManager` can also be assigned to a branch, in which case it will only manage the `Temporal`s of that branch.

In [14]:
branch.temporal_manager = TemporalManager()
branch.temporal_manager.set_swap_mode("loaded")

# swap mode of temporal will have changed:
print(heat_demand.input_flow.swap_mode)

loaded


### Cleaning up

When we are done with the project, we should close all open HDF files to be able to clean up the temporary directory:

In [15]:
# close files to be able to clean up temporary directory:
project.file_adapter.close_hdf_files()

# clean up temporary directory:
temp_dir.cleanup()